# Exp 2 — Distillation Ablation

Teacher: ResNet-18 (95% val accuracy, ~11M params)  
Student: DeiT-tiny, vanilla (no SPT/LSA)  

Three runs:
1. **Soft distillation** — τ=4, λ=0.5  
   L = 0.5·CE(student, y) + 0.5·τ²·KL(softmax(s/τ) ∥ softmax(t/τ))
2. **Hard distillation** — no dist token  
   L = 0.5·CE(student, y) + 0.5·CE(student, argmax(teacher))
3. **Hard distillation + distillation token** (DeiT)  
   L = 0.5·CE(cls_head, y) + 0.5·CE(dist_head, argmax(teacher))

**Rationale for τ=4, λ=0.5**:  
τ=4 softens the teacher's peaked output distribution enough to expose inter-class similarity
signal (e.g. 3 vs 8 are more confused than 1 vs 0) without collapsing to uniform.  
λ=0.5 splits the loss equally between ground truth and teacher knowledge — appropriate
when the teacher is strong (95%) and the student is significantly smaller.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL = "YOUR_GIT_REMOTE_HERE"   # TODO: set your git remote URL
REPO_DIR = "/content/repo"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "torchvision", "huggingface-hub",
                    "matplotlib", "seaborn", "scikit-learn"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(".", "..", ".."))

P3_SRC = os.path.join(REPO_DIR, "project3", "src")
P2_SRC = os.path.join(REPO_DIR, "project2", "src")
for p in [P3_SRC, P2_SRC]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(P3_SRC)

In [ ]:
# ── Data + device ────────────────────────────────────────────────────────────
import torch, numpy as np

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/cs148/data/dataset"  # TODO: adjust
else:
    DATA_DIR = os.path.join(REPO_DIR, "project2", "data", "dataset")

TEACHER_PATH = os.path.join(REPO_DIR, "project2", "checkpoints", "run10_final_9k", "best_model.pt")
BASE_SAVE = "/content/checkpoints" if IN_COLAB else "../checkpoints"

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Device:", device)

In [ ]:
# ── Shared hyperparameters ───────────────────────────────────────────────────
import argparse

COMMON = dict(
    data_dir=DATA_DIR,
    epochs=50,
    batch_size=64,
    lr=1e-3,
    weight_decay=0.05,
    warmup_epochs=15,
    drop_path_rate=0.1,
    img_size=128,
    repeat_aug=3,
    patience=0,
    num_workers=2,
    seed=42,
    synthetic_n=3000,
    val_fraction=0.15,
    use_spt=False,
    use_lsa=False,
    teacher_path=TEACHER_PATH,
    tau=4.0,
    lambda_dist=0.5,
)

In [ ]:
# ── Run 2a: Soft distillation (τ=4, λ=0.5) ──────────────────────────────────
import train
torch.manual_seed(42); np.random.seed(42)

args_soft = argparse.Namespace(
    **COMMON,
    save_dir=os.path.join(BASE_SAVE, "exp2a_soft_dist"),
    distillation="soft",
)
train.train(args_soft)

In [ ]:
# ── Run 2b: Hard distillation (no distillation token) ────────────────────────
# Reload train module to reset any global state
import importlib; importlib.reload(train)
torch.manual_seed(42); np.random.seed(42)

args_hard = argparse.Namespace(
    **COMMON,
    save_dir=os.path.join(BASE_SAVE, "exp2b_hard_dist"),
    distillation="hard",
)
train.train(args_hard)

In [ ]:
# ── Run 2c: Hard distillation + distillation token (DeiT) ────────────────────
importlib.reload(train)
torch.manual_seed(42); np.random.seed(42)

args_hard_dist = argparse.Namespace(
    **COMMON,
    save_dir=os.path.join(BASE_SAVE, "exp2c_hard_dist_token"),
    distillation="hard-dist",
)
train.train(args_hard_dist)

In [ ]:
# ── Results summary ──────────────────────────────────────────────────────────
import json

for name, save_dir in [
    ("soft",       os.path.join(BASE_SAVE, "exp2a_soft_dist")),
    ("hard",       os.path.join(BASE_SAVE, "exp2b_hard_dist")),
    ("hard-dist",  os.path.join(BASE_SAVE, "exp2c_hard_dist_token")),
]:
    log_path = os.path.join(save_dir, "training_log.json")
    if os.path.exists(log_path):
        with open(log_path) as f:
            history = json.load(f)
        best_va = max(e["val"]["acc"] for e in history)
        print(f"{name:15s}: best val_acc = {best_va:.4f}")

In [ ]:
# ── Download results (Colab only) ────────────────────────────────────────────
if IN_COLAB:
    import shutil
    from google.colab import files
    for name, save_dir in [
        ("exp2a_soft_dist", os.path.join(BASE_SAVE, "exp2a_soft_dist")),
        ("exp2b_hard_dist", os.path.join(BASE_SAVE, "exp2b_hard_dist")),
        ("exp2c_hard_dist_token", os.path.join(BASE_SAVE, "exp2c_hard_dist_token")),
    ]:
        zip_path = f"/content/{name}_results.zip"
        shutil.make_archive(f"/content/{name}_results", "zip", save_dir)
        files.download(zip_path)